In [1]:
import pandas as pd
import numpy as np
from sklearn.metrics import confusion_matrix
import seaborn as sns
import matplotlib.pyplot as plt
import torch.optim
from torch.nn.utils.rnn import pack_padded_sequence
import torch.nn as nn
from torch.utils.data import DataLoader
from torch.utils.data import Dataset
from torch.nn.utils.rnn import pad_sequence
from sklearn.model_selection import train_test_split

In [2]:
class TennisLSTM(nn.Module):

    def __init__(self, input_size):

        super().__init__()

        self.lstm = nn.LSTM(
            input_size=input_size,
            hidden_size=32,
            num_layers=1,
            batch_first=True,
            dropout=0.3
        )

        self.fc1 = nn.Linear(32,16)

        self.relu = nn.ReLU()

        self.dropout = nn.Dropout(0.3)

        self.fc2 = nn.Linear(16,2)

    def forward(self, x, lengths):

        packed = pack_padded_sequence(
            x,
            lengths.cpu(),
            batch_first=True,
            enforce_sorted=False
        )

        _, (hidden, _) = self.lstm(packed)

        x = hidden[-1]

        x = self.fc1(x)

        x = self.relu(x)

        x = self.dropout(x)

        x = self.fc2(x)

        return x

In [3]:

class TennisDataset(Dataset):

    def __init__(self, X):

        self.X = X

    def __len__(self):

        return len(self.X)

    def __getitem__(self, idx):

        sequence = torch.tensor(
            self.X[idx],
            dtype=torch.float32
        )
        length = len(sequence)

        return sequence, length
    

def collate_fn(batch):

    sequences = [item[0] for item in batch]

    lengths = torch.tensor(
        [item[1] for item in batch],
        dtype=torch.long
    )


    padded = pad_sequence(
        sequences,
        batch_first=True
    )

    return padded, lengths

In [ ]:
def pad_feature_sequences(sequences, max_length):

    n_features = sequences[0].shape[1]

    X = np.zeros((len(sequences), max_length, n_features),
                 dtype=np.float32)
    lengths = []
    for i, seq in enumerate(sequences):

        seq = seq[-max_length:]        
        lengths.append(len(seq))
        X[i,:len(seq)] = seq

    return X, np.array(lengths)

In [4]:
def prediction(X_test):
    test_dataset = TennisDataset(X_test)

    test_loader = DataLoader(
        test_dataset,
        batch_size=1,
        shuffle=False,
        collate_fn=collate_fn)
    model.eval()
    with torch.no_grad():
        for x, lengths in test_loader:
            x=x.to(device)  
            output = model(x,lengths.cpu())
            probs = torch.softmax(output,1)
        prob1 = probs[0,0].item()
        prob2 = probs[0,1].item()
        return prob1, prob2


In [5]:
def convert(score):
    mapping = {
        "0": 0,
        "15": 1,
        "30": 2,
        "40": 3,
        "AD": 4
    }
    score = str(score)
    if score in mapping:
        return mapping[score]
    return int(score)


In [7]:
sequence = []
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = TennisLSTM(input_size =10)
model.eval()
checkpoint = torch.load("best_tennis_lstm.pt", map_location=device)
model.load_state_dict(checkpoint["model_state_dict"])

model.to(device)

num_pts = int(input("Enter number of points played till now: "))



print("\nEnter each point as:")
print("Set1 Set2 Gm1 Gm2 P1_pts P2_pts Svr\n")

for i in range(num_pts):

    while True:

        local = input(f"Point {i+1}: ").split()

        if len(local) != 7:
            print("Please enter exactly 7 values.")
            continue

        local[0] = int(local[0])
        local[1] = int(local[1])
        local[2] = int(local[2])
        local[3] = int(local[3])

        local[4] = convert(local[4])
        local[5] = convert(local[5])

        local[6] = int(local[6])

        set_diff = local[0] - local[1]
        game_diff = local[2] - local[3]
        point_diff = local[4] - local[5]

        local.extend([set_diff, game_diff, point_diff])

        sequence.append(local)

        break
for seq in sequence:
    print(seq,'\n')
p1, p2 = prediction(sequence)

print(f"\nPlayer 1 win probability: {p1:.2%}")
print(f"Player 2 win probability: {p2:.2%}")


Enter each point as:
Set1 Set2 Gm1 Gm2 P1_pts P2_pts Svr

Please enter exactly 7 values.
[1, 0, 4, 3, 0, 0, 2, 1, 1, 0] 

[1, 0, 4, 3, 1, 0, 2, 1, 1, 1] 

[1, 0, 4, 3, 1, 1, 2, 1, 1, 0] 

[1, 0, 4, 3, 2, 1, 2, 1, 1, 1] 

[1, 0, 4, 3, 2, 3, 2, 1, 1, -1] 


Player 1 win probability: 67.56%
Player 2 win probability: 32.44%
